In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE

In [29]:
# Load CSV
df = pd.read_csv("deduplicated_clean_data.csv")

In [30]:
# Clean Contact Function column
df['Contact Function'] = df['Contact Function'].astype(str)
unknown_df = df[df['Contact Function'].str.lower().str.strip() == 'unknown'].copy()
known_df = df[~(df['Contact Function'].str.lower().str.strip() == 'unknown')].copy()

In [31]:
# Feature engineering
def engineer_features(df):
    df['Job_Title_Clean'] = df['Job Title'].fillna('').astype(str).str.lower().str.strip()
    df['Org_Clean'] = df['Organization'].fillna('').astype(str).str.lower().str.strip()
    df['Title_Org_Combo'] = df['Job_Title_Clean'] + ' at ' + df['Org_Clean']
    df['Country_Clean'] = df['Physical Country'].fillna('').astype(str).str.lower().str.strip()

    for col in ['Do Not Call Flag', 'Do Not Email Flag', 'Do Not Mail Flag']:
        df[col] = (
            df[col].fillna('0').astype(str).str.lower()
            .map({'y': 1, 'n': 0, '1': 1, '0': 0})
            .fillna(0).astype(int)
        )

    df['Total_Do_Not_Flags'] = df[['Do Not Call Flag', 'Do Not Email Flag', 'Do Not Mail Flag']].sum(axis=1)
    return df

known_df = engineer_features(known_df)
unknown_df = engineer_features(unknown_df)

In [32]:
# Drop rare classes (fewer than 2 samples)
class_counts = known_df['Contact Function'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
known_df = known_df[known_df['Contact Function'].isin(valid_classes)].copy()

In [33]:
# Encode categorical features
features = ['Title_Org_Combo', 'Country_Clean', 'Total_Do_Not_Flags']
le_dict = {}
for col in ['Title_Org_Combo', 'Country_Clean']:
    le = LabelEncoder()
    known_df[col] = le.fit_transform(known_df[col])
    le_dict[col] = le

    # Handle unknowns
    unknown_df[col] = unknown_df[col].map(lambda x: x if x in le.classes_ else 'unknown')
    if 'unknown' not in le.classes_:
        le.classes_ = np.append(le.classes_, 'unknown')
    unknown_df[col] = le.transform(unknown_df[col])

# Encode target
target_le = LabelEncoder()
known_df['Contact Function Encoded'] = target_le.fit_transform(known_df['Contact Function'])



In [34]:
# Split into features and labels
X = known_df[features]
y = known_df['Contact Function Encoded']

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [35]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42
)

In [36]:
# Train model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, random_state=42))])

In [37]:
# Evaluate
y_pred = pipeline.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.unique(y_test),
    target_names=target_le.inverse_transform(np.unique(y_test)),
    zero_division=0
))

✅ Accuracy: 0.6203358208955224

📊 Classification Report:
                               precision    recall  f1-score   support

                 board member       0.63      0.59      0.61        56
                          ceo       0.73      1.00      0.84        56
                          cio       0.56      0.59      0.57        56
           consultant - field       0.53      0.36      0.43        56
        consultant - research       0.63      0.54      0.58        57
  consultant - sr. management       0.46      0.64      0.53        56
              department head       0.60      0.68      0.64        57
                     director       0.56      0.77      0.65        57
                    executive       0.29      0.36      0.32        56
            financial advisor       0.80      0.21      0.33        57
independent financial advisor       0.61      0.68      0.64        57
           investment analyst       0.78      0.70      0.74        56
                  m

In [38]:
# Predict unknowns
if len(unknown_df) > 0:
    unknown_df['Predicted Contact Function'] = target_le.inverse_transform(
        pipeline.predict(unknown_df[features])
    )
    unknown_df[['Job Title', 'Organization', 'Physical Country',
                'Do Not Call Flag', 'Do Not Email Flag', 'Do Not Mail Flag',
                'Predicted Contact Function']].to_csv("pred_contact_function_final.csv", index=False)
    print("Predictions saved to 'pred_contact_function_final.csv'")
else:
    print(" No unknown Contact Function rows found.")

Predictions saved to 'pred_contact_function_final.csv'


In [39]:
# Optional: Save cleaned known data too
known_df.to_csv("cleaned_known_contact_function.csv", index=False)

In [40]:
import pandas as pd

# Step 1: Load the original full dataset
full_df = pd.read_csv("deduplicated_clean_data.csv")

# Step 2: Load the predicted Contact Function results
predicted_df = pd.read_csv("pred_contact_function_final.csv")

# Step 3: Identify rows where Contact Function is 'unknown' or NaN
mask_unknown = (
    full_df['Contact Function'].isna() | 
    (full_df['Contact Function'].str.lower().str.strip() == 'unknown')
)

# Step 4: Replace only those unknown Contact Functions with predictions
full_df.loc[mask_unknown, 'Contact Function'] = predicted_df['Predicted Contact Function'].values

# Step 5: Save to a final CSV
full_df.to_csv("final_combined_contact_function.csv", index=False)
print("✅ Final combined CSV saved as 'final_combined_contact_function.csv'")

✅ Final combined CSV saved as 'final_combined_contact_function.csv'


In [44]:
# Load your dataset
df = pd.read_csv("final_combined_contact_function.csv")  # or use your original file

# Count frequency of each value in 'Contact Function'
value_counts = df['Contact Function'].value_counts()

# Display the counts
print("✅ Contact Function value counts:\n")
print(value_counts)

✅ Contact Function value counts:

Contact Function
financial advisor                336
independent financial advisor    307
director                         105
migrated k2                       38
consultant - sr. management       36
sales assistant                   31
product manager / counsellors     20
investment analyst                20
department head                   15
ceo                               13
consultant - field                11
partner                           11
portfolio manager                  9
consultant - research              9
board member                       9
operations                         8
cio                                7
sales representative               3
executive                          3
legal                              1
marketing                          1
broker/dealers                     1
consultant - research/field        1
Name: count, dtype: int64


In [43]:


# Load your dataset
df = pd.read_csv("deduplicated_clean_data.csv")  # or use your original file

# Count frequency of each value in 'Contact Function'
value_counts = df['Contact Function'].value_counts()

# Display the counts
print("✅ Contact Function value counts:\n")
print(value_counts)


✅ Contact Function value counts:

Contact Function
unknown                          547
financial advisor                282
independent financial advisor     51
investment analyst                19
director                          18
department head                   15
portfolio manager                  7
ceo                                6
cio                                6
consultant - sr. management        6
product manager / counsellors      6
board member                       5
consultant - research              5
migrated k2                        5
operations                         3
partner                            2
sales representative               2
sales assistant                    2
consultant - field                 2
executive                          2
legal                              1
marketing                          1
broker/dealers                     1
consultant - research/field        1
Name: count, dtype: int64


In [47]:
# Load both CSV files
df_orig = pd.read_csv("deduplicated_clean_data.csv")
df_pred = pd.read_csv("final_combined_contact_function.csv")

# Merge both dataframes on Global ID
merged = pd.merge(
    df_orig[['Contact Global ID', 'Contact Function']],
    df_pred[['Contact Global ID', 'Contact Function']],
    on='Contact Global ID',
    suffixes=('_original', '_predicted')
)

# Filter rows where the value changed
diffs = merged[merged['Contact Function_original'] != merged['Contact Function_predicted']]

# Display results
print("🔍 Changed Contact Function values based on Global ID:\n")
print(diffs[['Contact Global ID', 'Contact Function_original', 'Contact Function_predicted']])

# Save to CSV if needed
diffs.to_csv("contact_function_changes.csv", index=False)
print("\n✅ Comparison saved to 'contact_function_changes.csv'")

🔍 Changed Contact Function values based on Global ID:

      Contact Global ID Contact Function_original  \
1              15782580                   unknown   
2              15782250                   unknown   
4              16615984                   unknown   
13             17880899                   unknown   
16             18542590                   unknown   
...                 ...                       ...   
1010           17926256                   unknown   
1011           16428938                   unknown   
1012           16428938                   unknown   
1015           19892473                   unknown   
1016           19892473                   unknown   

         Contact Function_predicted  
1                   sales assistant  
2     independent financial advisor  
4                 financial advisor  
13    independent financial advisor  
16    independent financial advisor  
...                             ...  
1010  independent financial advisor  
1011